# 01 Difference-in-Differences

DiD is the workhorse research design for causal inference when randomization is impossible but you have *panel* data: same units observed before and after a policy change, plus an untreated comparison group.


## Table of Contents
- [The 2x2 DiD intuition](#twobytwo)
- [Clean simulation: recover a known effect](#sim)
- [DiD as an OLS regression](#did-as-ols)
- [Parallel trends (the load-bearing assumption)](#parallel)
- [Apply to a county panel with a synthesized policy](#county-panel)
- [What can break DiD](#what-breaks)
- [Checkpoint (Self-Check)](#checkpoint-self-check)
- [Solutions (Reference)](#solutions-reference)


## Why This Notebook Matters
Most of applied econ does not have randomized experiments. Every macro question, most policy questions, most labor and health questions are observational. DiD is the dominant design for credibly answering causal questions in those settings — minimum-wage effects, medicaid expansions, environmental regulations, school-system reforms, the list is long.

The mechanic of DiD is two subtractions: the change in outcome for the treated group minus the change in outcome for the control group. That subtraction differences out (a) any time-invariant differences between groups and (b) any common time trends. What is left is, under one heroic assumption (parallel trends), the causal effect of the treatment.

You will learn:
- the two-by-two DiD identity by hand,
- how DiD pops out of an OLS regression with the right interaction,
- how to simulate panel data with a known treatment effect and recover it,
- how to apply DiD to the bundled Census county panel after synthesizing a state-level policy,
- the parallel-trends assumption and how to check it visually,
- what fails DiD: anticipation, spillovers, and treatment effects that vary over time.

## Prerequisites (Quick Self-Check)
- §02 (regression) — comfortable with multiple regression and interaction terms.
- The previous notebook (`00_predictive_vs_causal_dags.ipynb`).

## What You Will Produce
- A simulation that recovers a known $\beta_{DiD}$ within sampling error.
- A DiD regression with clustered standard errors on a panel of US counties.
- A parallel-trends plot for your treatment and control groups.

## Common Pitfalls
- Reporting a DiD coefficient without checking parallel trends visually first.
- Using OLS standard errors on panel data — they will badly understate uncertainty. Cluster at the level of the treatment (here, the state).
- Mistaking a level difference between groups for a treatment effect. DiD differences out the level; the *change* is what matters.
- Including post-treatment time-varying covariates that are themselves affected by treatment (a collider trap).

## Quick Fixes (When You Get Stuck)
- If the DiD coefficient is far from the true effect in your simulation, check that you encoded post and treated as 0/1 the way you expect.
- If clustering changes SE dramatically, that is normal — the un-clustered SE was wrong.

## Matching Guide
- `docs/guides/05_causal_inference/01_difference_in_differences.md`


<a id="environment-bootstrap"></a>
## Environment Bootstrap


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    p = start
    for _ in range(8):
        if (p / 'src').exists() and (p / 'docs').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root. Start Jupyter from the repo root.')


PROJECT_ROOT = find_repo_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
SAMPLE_DIR = DATA_DIR / 'sample'
PROJECT_ROOT


<a id="twobytwo"></a>
## The 2x2 DiD intuition

The cleanest DiD setup: one treatment group, one control group, observed at one pre-period and one post-period. Four numbers:

|  | Pre | Post |
|---|---|---|
| **Treated** | $\bar Y_{T,0}$ | $\bar Y_{T,1}$ |
| **Control** | $\bar Y_{C,0}$ | $\bar Y_{C,1}$ |

$$
\hat \beta_{DiD} = (\bar Y_{T,1} - \bar Y_{T,0}) - (\bar Y_{C,1} - \bar Y_{C,0})
$$

Equivalently:

$$
\hat \beta_{DiD} = (\bar Y_{T,1} - \bar Y_{C,1}) - (\bar Y_{T,0} - \bar Y_{C,0})
$$

Same number, different interpretation: the change in the gap between groups, or the difference in the changes within groups.

**What this differences out:**
- Any time-invariant difference between treated and control (e.g., one is rural, one is urban).
- Any common time trend that affects both groups equally (e.g., national recession).

**What this does *not* difference out:**
- Time-varying *group-specific* shocks. Those are the lurking threat — and the parallel-trends assumption is exactly the assumption that they don't exist.


<a id="sim"></a>
## Clean simulation: recover a known effect

Let's build panel data with a known treatment effect $\tau = -1.5$ and check that DiD recovers it.


### Your Turn (1): Simulate the panel


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
n_units = 200
T_periods = 10
treatment_period = 5
tau = -1.5  # true treatment effect

# Half the units are treated
unit_ids = np.arange(n_units)
is_treated = (unit_ids < n_units // 2).astype(int)

# Each unit has a baseline level (so groups have different means even before treatment)
unit_baseline = rng.normal(loc=0, scale=2.0, size=n_units)
# Treatment group has a slightly higher baseline (this is differenced out by DiD)
unit_baseline += is_treated * 1.0

# Within-unit shocks are AR(1): today's noise is correlated with yesterday's.
# This is realistic for panel data and is what makes clustering necessary.
rho = 0.7
shocks = np.zeros((n_units, T_periods))
for u in unit_ids:
    shocks[u, 0] = rng.normal(scale=1.0)
    for t in range(1, T_periods):
        shocks[u, t] = rho * shocks[u, t - 1] + rng.normal(scale=1.0)

rows = []
for t in range(T_periods):
    common_trend = 0.3 * t  # both groups trend up over time
    for u in unit_ids:
        post = int(t >= treatment_period)
        treat = is_treated[u]
        treat_effect = tau if (treat == 1 and post == 1) else 0
        y = unit_baseline[u] + common_trend + treat_effect + shocks[u, t]
        rows.append({'unit': u, 'time': t, 'treated': treat, 'post': post, 'y': y})

panel = pd.DataFrame(rows)
panel.head()


### Your Turn (2): Compute DiD by hand from four cell means


In [ ]:
cell_means = panel.groupby(['treated', 'post'])['y'].mean().unstack()
cell_means.index = ['Control', 'Treated']
cell_means.columns = ['Pre', 'Post']
print(cell_means.round(3))

did_byhand = (cell_means.loc['Treated', 'Post'] - cell_means.loc['Treated', 'Pre']) \
           - (cell_means.loc['Control', 'Post'] - cell_means.loc['Control', 'Pre'])
print(f'\nDiD by hand: {did_byhand:.3f}')
print(f'True tau:    {tau}')


<a id="did-as-ols"></a>
## DiD as an OLS regression

The 2x2 DiD is exactly the coefficient on the interaction term in:

$$
y_{u,t} = \alpha + \beta_{T} \cdot \text{treated}_u + \beta_{P} \cdot \text{post}_t + \beta_{DiD} \cdot (\text{treated}_u \times \text{post}_t) + u_{u,t}
$$

Why this formulation matters: it scales naturally to multiple time periods, multiple control variables, and clustered standard errors. The 4-cells-mean version was a special case.


### Your Turn (1): Run the DiD regression


In [ ]:
panel['treated_x_post'] = panel['treated'] * panel['post']
X = sm.add_constant(panel[['treated', 'post', 'treated_x_post']], has_constant='add')
did_res = sm.OLS(panel['y'], X).fit()
print(did_res.summary().tables[1])


### Your Turn (2): Cluster standard errors at the unit level

Repeated observations of the same unit are not independent. Cluster SE at `unit` (or, for state-level treatments, cluster at `state`).


In [ ]:
# TODO: refit with cov_type='cluster' clustered on unit.
# Hint: cov_kwds={'groups': panel['unit']}
did_clu = sm.OLS(panel['y'], X).fit(
    cov_type='cluster', cov_kwds={'groups': panel['unit']}
)

comparison = pd.DataFrame({
    'coef': did_res.params,
    'naive_SE': did_res.bse,
    'clustered_SE': did_clu.bse,
})
comparison['ratio'] = comparison['clustered_SE'] / comparison['naive_SE']
comparison.round(4)


### Your Turn (3): Interpretation

The coefficient on `treated_x_post` should be near $\tau = -1.5$. The SE comparison is more interesting than 'larger or smaller':

- For `const` and `treated` (parameters identified from **between-unit** variation), naive SE is far too small because it pretends 200×10 = 2000 independent observations exist when there are really 200 units. Clustered SE corrects this and inflates substantially.
- For `post` and `treated_x_post` (parameters identified from **within-unit** variation, i.e., changes over time within a unit), the direction depends on the within-unit shock structure. With AR(1) shocks (positive serial correlation), within-unit *differences* are actually less noisy than naive iid would predict, so clustered SE can be smaller.

Write 4–6 sentences:
- What does the `treated` coefficient by itself measure?
- What does the `post` coefficient by itself measure?
- Why is `treated_x_post` the right place to read the causal effect?
- The DiD coefficient itself didn't change between naive and clustered. So why bother clustering?


In [ ]:
notes = """
...
"""
print(notes)


<a id="parallel"></a>
## Parallel trends (the load-bearing assumption)

DiD identifies the treatment effect only if, in the absence of treatment, the treated and control groups would have moved in parallel. You cannot test this with the post-treatment data (you do not observe the counterfactual). What you *can* do is check that the **pre-treatment** trends are parallel and treat that as evidence the assumption is plausible.


### Your Turn (1): Plot pre-treatment trends


In [ ]:
trend = panel.groupby(['time', 'treated'])['y'].mean().unstack()
trend.columns = ['Control', 'Treated']

fig, ax = plt.subplots(figsize=(9, 4))
trend.plot(ax=ax, marker='o')
ax.axvline(treatment_period - 0.5, color='red', linestyle='--', label='treatment')
ax.set_xlabel('time')
ax.set_ylabel('mean y')
ax.set_title('Group means over time (pre-period parallel; post-period diverges)')
ax.legend()
plt.tight_layout()


<a id="county-panel"></a>
## Apply to a county panel with a synthesized policy

We use the bundled `census_county_panel_sample.csv` (3 states × 3 counties × 9 years) and synthesize a state-level policy in California (state 06) that takes effect in 2018. The 'effect' we hard-code is a 1.5 percentage-point reduction in unemployment_rate. DiD should recover something close to that.

Reality check: 9 years × 9 counties = 81 observations is small. The point is the *workflow*; SE will be wide.


### Your Turn (1): Load and synthesize


In [ ]:
panel_path = SAMPLE_DIR / 'census_county_panel_sample.csv'
cdf = pd.read_csv(panel_path, dtype={'state': str, 'county': str, 'fips': str})
cdf['year'] = cdf['year'].astype(int)

# Synthesize: California (state 06) gets a 1.5 pp reduction in unemployment starting in 2018.
policy_state = '06'
policy_year = 2018
true_effect_pp = -1.5

cdf['treated'] = (cdf['state'] == policy_state).astype(int)
cdf['post'] = (cdf['year'] >= policy_year).astype(int)
cdf['treated_x_post'] = cdf['treated'] * cdf['post']

# Apply the synthesized effect to the unemployment_rate column.
# The unemployment_rate is a fraction (0–1); we add the effect in pp = effect / 100.
cdf['unemployment_rate_synth'] = cdf['unemployment_rate'] + (cdf['treated_x_post'] * true_effect_pp / 100.0)

cdf[['state', 'fips', 'year', 'treated', 'post', 'unemployment_rate', 'unemployment_rate_synth']].head(10)


### Your Turn (2): Visual parallel-trends check


In [ ]:
trend_real = cdf.groupby(['year', 'treated'])['unemployment_rate_synth'].mean().unstack() * 100.0
trend_real.columns = ['Non-CA', 'CA (treated)']

fig, ax = plt.subplots(figsize=(9, 4))
trend_real.plot(ax=ax, marker='o')
ax.axvline(policy_year - 0.5, color='red', linestyle='--', label='policy 2018')
ax.set_ylabel('unemployment rate (%)')
ax.set_title('Mean county unemployment by group, synthesized policy in CA from 2018')
ax.legend()
plt.tight_layout()


### Your Turn (3): DiD regression with state-clustered SE


In [ ]:
# Note: with only 3 states, clustered SE will be unreliable. We do it anyway because
# the workflow is right; the point of the toy panel is to show mechanics.
X = sm.add_constant(cdf[['treated', 'post', 'treated_x_post']], has_constant='add')
y = cdf['unemployment_rate_synth'] * 100.0  # convert to pp for readability

res_did = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': cdf['state']})
print(res_did.summary().tables[1])
print(f'\nTrue treatment effect (pp): {true_effect_pp}')


### Your Turn (4): Why the clustered SE is too narrow with 3 clusters

Clustered SE has good asymptotic properties as the number of clusters grows. With 3 clusters you are well below the rule-of-thumb minimum (~30). Real DiD work either uses many states or applies a small-cluster correction (wild cluster bootstrap). For this notebook, recognize the limitation; do not put weight on the p-value.


<a id="what-breaks"></a>
## What can break DiD

Things that invalidate the parallel-trends assumption or the inference:

1. **Anticipation.** Units that anticipate treatment may change behavior before the announced date. The DiD coefficient then mixes the actual effect with the anticipatory adjustment.
2. **Spillovers.** If treating California affects Nevada (workers commute, firms relocate), then 'control' is not a true counterfactual.
3. **Differential trends.** If the treated group was already on a different trajectory before treatment, DiD attributes the divergence to the policy.
4. **Heterogeneous and time-varying effects.** Standard DiD assumes the effect is constant across units and time. Recent work (Sun and Abraham 2021, Callaway and Sant'Anna 2021) shows two-way fixed effects DiD can give absurd answers under heterogeneity.
5. **Few clusters.** Standard clustered SE break down when the number of clusters is small. Use wild cluster bootstrap.
6. **Compositional change.** If the treatment changes who appears in the data (selection), DiD measures both the policy effect and the selection effect.

Check the parallel-trends plot first. Then ask which of these threats is most plausible for your application. Then, ideally, address them — placebo tests, event-study plots, alternative control groups.


<a id="checkpoint-self-check"></a>
## Checkpoint (Self-Check)


In [ ]:
# Sanity asserts (uncomment after the cells above run cleanly):
# assert abs(did_byhand - tau) < 0.2
# assert abs(did_res.params['treated_x_post'] - tau) < 0.2
# assert did_clu.bse['treated'] > did_res.bse['treated']  # clustered SE on between-unit terms IS larger
# assert abs(res_did.params['treated_x_post'] - true_effect_pp) < 0.5
...


## Extensions (Optional)
- Run an **event-study** version: include leads and lags of treatment as separate dummies. Plot the lead coefficients to inspect pre-treatment trends formally.
- Add a **placebo test**: pretend the treatment happened in 2016 (no actual policy that year) and re-run DiD. The placebo coefficient should be near zero.
- Modify the synthetic-policy example so the effect is *delayed* (kicks in fully only by 2020). What does the standard DiD coefficient measure now?
- Use the `linearmodels` package's `PanelOLS` with two-way fixed effects (unit and year) instead of the two dummies.


## Reflection
- DiD differences out two things and assumes one big thing. What are they?
- For a question in your own domain (work or otherwise) that you would *like* to answer causally, what would the treatment, control, pre-period, and post-period be? Could you defend parallel trends?


<a id="solutions-reference"></a>
## Solutions (Reference)

<details><summary>Solution: Cluster SE</summary>

```python
did_clu = sm.OLS(panel['y'], X).fit(
    cov_type='cluster', cov_kwds={'groups': panel['unit']}
)
```

On this simulation you should see:
- `treated` and `const`: clustered SE roughly **2x larger** than naive (between-unit comparisons; naive SE wildly understates uncertainty).
- `post` and `treated_x_post`: clustered SE roughly **0.75x of naive** (within-unit comparisons; AR(1) within-unit shocks make within-unit differences *less* noisy than iid would predict).

The lesson is not 'clustered is always bigger.' The lesson is: naive SE assumes independence and gives the wrong answer when observations cluster; clustered SE gives the right answer for the unit-level dependence structure, whatever that direction turns out to be.

</details>

<details><summary>Solution: Coefficient interpretations</summary>

- `treated`: time-invariant gap between groups (level difference, not the policy effect).
- `post`: common time trend; what *both* groups would have done after the cutoff if neither were treated.
- `treated_x_post`: the *additional* change that happened to the treated group, on top of the common time trend. This is $\beta_{DiD}$.

</details>

<details><summary>Solution: Why clustering matters even though the point estimate didn't change</summary>

OLS inference assumes independent observations. Repeated measurements of the same unit are not independent: they share unit-specific shocks. Naive SE under-estimates uncertainty for between-unit comparisons (treating 200 units as 2000 obs). Clustered SE corrects the variance of the *estimator*. The estimator itself (the coefficient) is unaffected — only your confidence in it changes.

</details>
